# Сбор данных с Web-scraping и API для гуманитарных исследований
## Блок 1. API
## Занятие 1. Как работает запрос. Первый API

Любой сбор данных из интернета, будь то скачивание страницы или обращение к API, сводится к обмену сообщениями между двумя программами: одна отправляет запрос, другая присылает ответ. На этом занятии разбирается, как устроен такой обмен, и делаются первые запросы — сначала к обычной странице Википедии, затем к её API.

**Материал:** русская Википедия и её программный интерфейс MediaWiki API.

**Исследовательская задача.** Википедию можно рассматривать как срез культурной памяти: о ком и о чём пишут, сколько и как подробно. К концу занятия вы соберёте из API два набора данных и сохраните их в CSV:

1. таблицу заметности писателей, философов или других фигур вашей области в русской Википедии — простой культуромический показатель, который можно сравнивать;
2. список статей из категории Википедии — заготовку корпуса для собственного исследования: музеи города, писатели эпохи, памятники архитектуры.

Не менее важно научиться оценивать, что на самом деле стоит за этими числами и в каких случаях на них нельзя полагаться.

**Что предстоит освоить:**
- читать адрес страницы (URL) и разбирать его на части;
- отправлять запросы библиотекой `requests` и понимать ответ сервера: статус-код, заголовки, содержимое;
- представляться серверу и соблюдать правила сайта;
- обращаться к API и разбирать ответ в формате JSON;
- превращать ответ в таблицу `pandas` и сохранять её в CSV.

---

Ячейки запускаются сверху вниз: каждая часть опирается на предыдущую. Задания для самостоятельной работы отмечены ✏️; код пишется в ячейке под заданием на месте комментария `# ВАШ КОД`. 

Блоки повторения материала курса «Цифровые технологии в профессиональной деятельности» отмечены 🔁. 

Задания с 📨 предполагают ваш выбор и рассуждение: их результат отправляется в телеграм-чат курса, чтобы его можно было обсудить вместе.

---
## Часть 0. Импорт библиотек

Все библиотеки курса перечислены в файле `requirements.txt` в корне репозитория. Если окружение создано по инструкции со страницы «Установка», они уже установлены.

In [1]:
# Установка, если библиотек нет (раскомментировать и выполнить один раз)
# %pip install requests pandas

In [2]:
import re
import time
from urllib.parse import urlparse, parse_qs

import pandas as pd
import requests

---
## Часть 1. Как устроен обмен данными в вебе

### Клиент и сервер

Когда в браузере открывается страница, происходит обмен двумя сообщениями:

1. **Клиент** (браузер или программа на Python) отправляет **запрос** (request): «пришли мне такую-то страницу».
2. **Сервер** (компьютер, на котором хранится сайт) присылает **ответ** (response): саму страницу или сообщение об ошибке.

Правила, по которым составляются эти сообщения, называются **HTTP** (HyperText Transfer Protocol). Буква *s* в `https` означает, что сообщения шифруются; на устройство запроса это не влияет.

Браузер делает всё это незаметно и показывает уже готовую страницу. Программа на Python отправляет те же запросы, но получает ответ «как есть» — в виде текста или данных, с которыми дальше можно работать.

### Из чего состоит адрес

Любой запрос начинается с адреса — **URL** (Uniform Resource Locator). Пример — адрес поисковой выдачи Википедии:

`https://ru.wikipedia.org/w/index.php?search=Пушкин&ns0=1`

| Часть | Пример | Что означает |
|---|---|---|
| Схема (scheme) | `https` | по какому протоколу идёт обмен |
| Хост (host) | `ru.wikipedia.org` | к какому серверу обращаемся |
| Путь (path) | `/w/index.php` | какой ресурс на сервере нужен |
| Параметры (query) | `search=Пушкин&ns0=1` | уточнения запроса: пары `ключ=значение`, разделённые `&` |

Параметры отделяются от пути знаком `?`. В параметрах обычно передаётся то, что пользователь ввёл в поисковую строку или выбрал в фильтрах. Модуль `urllib.parse` из стандартной библиотеки умеет разбирать адрес на части.

In [3]:
url = "https://ru.wikipedia.org/w/index.php?search=Пушкин&ns0=1"

parts = urlparse(url)

# Что вернула функция: объект, в котором адрес разобран на части
parts

ParseResult(scheme='https', netloc='ru.wikipedia.org', path='/w/index.php', params='', query='search=Пушкин&ns0=1', fragment='')

У объекта `parts` есть отдельное поле для каждой части адреса. К полю обращаются через точку.

In [4]:
print("Схема:", parts.scheme)
print("Хост:", parts.netloc)
print("Путь:", parts.path)
print("Параметры:", parts.query)

Схема: https
Хост: ru.wikipedia.org
Путь: /w/index.php
Параметры: search=Пушкин&ns0=1


In [5]:
# parse_qs превращает строку параметров в словарь
# значения — списки, потому что один ключ может встретиться в адресе несколько раз
parse_qs(parts.query)

{'search': ['Пушкин'], 'ns0': ['1']}

Если скопировать такой адрес из адресной строки браузера, кириллица в нём превратится в последовательности вроде `%D0%9F`. Это **URL-кодирование**: в адресе допустим только ограниченный набор символов, остальные записываются кодами. `parse_qs` раскодирует их автоматически.

#### ✏️ Задание 1

Откройте в браузере сайт с поиском из вашей области — Национальный корпус русского языка, каталог библиотеки, сайт музея — и выполните любой поиск. Скопируйте адрес страницы с результатами и разберите его: выведите хост, путь и словарь параметров. Какие параметры удаётся понять без документации? В каком из них записан ваш поисковый запрос?

Если под рукой нет такого сайта, возьмите адрес поиска по Википедии из переменной ниже: он записан так, как копируется из браузера.

In [6]:
my_url = "https://ru.wikipedia.org/w/index.php?search=%D0%A2%D0%B5%D0%BA%D1%81%D1%82%D0%BE%D0%BB%D0%BE%D0%B3%D0%B8%D1%8F&title=%D0%A1%D0%BB%D1%83%D0%B6%D0%B5%D0%B1%D0%BD%D0%B0%D1%8F:%D0%9F%D0%BE%D0%B8%D1%81%D0%BA&ns0=1"

# ВАШ КОД

---
## Часть 2. Первый запрос

### Библиотека requests

Для отправки HTTP-запросов из Python обычно используется библиотека `requests`. Главная её функция — `requests.get()`: она отправляет запрос по адресу и возвращает объект ответа.

Слово *get* здесь не случайно. Каждый HTTP-запрос сопровождается **методом** — глаголом, по которому сервер понимает, что клиент хочет сделать:

| Метод | Что делает | Пример |
|---|---|---|
| `GET` | получить данные | открыть страницу, выполнить поиск |
| `POST` | отправить данные | отправить форму, загрузить файл |
| `PUT`, `PATCH` | изменить данные | отредактировать запись |
| `DELETE` | удалить данные | удалить запись |

Для сбора данных почти всегда достаточно `GET`. Первый запрос — к статье Википедии.

In [7]:
page_url = "https://ru.wikipedia.org/wiki/Цифровые_гуманитарные_науки"

response = requests.get(page_url, timeout=10)

response

<Response [403]>

Параметр `timeout=10` ограничивает ожидание десятью секундами: если сервер не ответил, программа не будет ждать бесконечно. Его стоит указывать в каждом запросе.

Ответ состоит из трёх частей:

1. **статус-код** — трёхзначное число, итог запроса;
2. **заголовки** (headers) — служебные сведения об ответе;
3. **тело** (body) — само содержимое: HTML-страница, данные, картинка.

Ниже — каждая из них по отдельности.

In [8]:
# 1. Статус-код
response.status_code

403

In [9]:
# 2. Заголовки ответа: все сразу
response.headers

{'content-length': '126', 'content-type': 'text/plain', 'x-request-id': '97f5df74-8a33-40ef-ace8-ab105f0022d9', 'server': 'HAProxy', 'x-cache': 'cp3070 int', 'x-cache-status': 'int-tls', 'x-analytics': ''}

Заголовки устроены как словарь: название заголовка — ключ, его содержимое — значение. Большинство из них служебные и нужны браузеру. Для сбора данных чаще всего нужен `Content-Type` — тип содержимого ответа. Его можно получить по названию, как значение словаря по ключу.

In [10]:
# Один заголовок по названию
response.headers["Content-Type"]

'text/plain'

In [11]:
# 3. Тело ответа в виде текста: сначала — сколько в нём символов
len(response.text)

126

In [12]:
print(response.text[:500])

Please set a user-agent and respect our robot policy https://w.wiki/4wJS. See also https://phabricator.wikimedia.org/T400119.



Статьи в ответе нет. Сервер вернул код `403`, а в теле — короткое текстовое сообщение: Википедия просит указать User-Agent и соблюдать правила для роботов. Почему так произошло, объясняется в следующей части; сначала — о статус-кодах.

### Статус-коды

По первой цифре кода можно определить тип результата:

| Код | Значение | Что делать |
|---|---|---|
| `200 OK` | запрос выполнен | работать с ответом |
| `301`, `302` | ресурс переехал | `requests` сам переходит по новому адресу |
| `400 Bad Request` | запрос составлен неправильно | проверить адрес и параметры |
| `403 Forbidden` | доступ запрещён | выяснить, на каких условиях сайт разрешает запросы |
| `404 Not Found` | ресурса нет | проверить адрес |
| `429 Too Many Requests` | слишком частые запросы | сделать паузу |
| `500`, `503` | ошибка на сервере | повторить позже |

Коды на `2` означают успех, на `3` — перенаправление, на `4` — ошибку клиента, на `5` — ошибку сервера.

Сайты меняют свои правила. Если у вас пришёл код `200`, Википедия в этот момент не требует User-Agent; следующая часть всё равно нужна, потому что многие другие сайты его требуют.

---
## Часть 3. Как представиться серверу

Заголовки есть не только у ответа, но и у запроса. Ниже — заголовки, которые `requests` отправил серверу.

In [13]:
# response.request — запрос, на который пришёл этот ответ
response.request.headers

{'User-Agent': 'python-requests/2.32.5', 'Accept-Encoding': 'gzip, deflate', 'Accept': '*/*', 'Connection': 'keep-alive'}

In [14]:
# Один заголовок запроса: подпись программы
response.request.headers["User-Agent"]

'python-requests/2.32.5'

В заголовке **User-Agent** указано, какая программа отправила запрос. Браузер записывает туда своё название и версию, `requests` — строку `python-requests/…`.

Многие сайты блокируют запросы с такой безликой подписью. У проектов Wikimedia есть [правила для User-Agent](https://foundation.wikimedia.org/wiki/Policy:Wikimedia_Foundation_User-Agent_Policy): в запросах скрипта должно быть указано его название и способ связаться с автором, иначе скрипт могут заблокировать без предупреждения. Выдавать скрипт за браузер по этим правилам тоже нельзя: такие запросы считаются злонамеренными.

Заголовки запроса передаются словарём в параметре `headers`.

In [15]:
# Название скрипта, версия и в скобках контакт: адрес репозитория курса
HEADERS = {
    "User-Agent": "miniDH-scraping-course/1.0 (https://github.com/Minidegree-DH/miniDH-scraping)"
}

HEADERS

{'User-Agent': 'miniDH-scraping-course/1.0 (https://github.com/Minidegree-DH/miniDH-scraping)'}

In [16]:
# Тот же запрос, но теперь со своими заголовками
response = requests.get(page_url, headers=HEADERS, timeout=10)

response.status_code

200

In [17]:
# Проверка: с какой подписью ушёл запрос
response.request.headers["User-Agent"]

'miniDH-scraping-course/1.0 (https://github.com/Minidegree-DH/miniDH-scraping)'

In [18]:
# Что прислал сервер теперь
response.headers["Content-Type"]

'text/html; charset=UTF-8'

In [19]:
print(response.text[:500])

<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-enabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpr


Теперь сервер прислал страницу: `Content-Type` равен `text/html`, а в теле лежит HTML-код — тот же, что браузер превращает в оформленный текст с картинками. Как доставать из HTML нужные фрагменты, разбирается на занятии 3. На этом занятии речь пойдёт о другом пути к данным — об API.

#### ✏️ Задание 2

Составьте собственную строку User-Agent по образцу: название вашего скрипта, версия и в скобках контакт — адрес вашего репозитория на GitHub или корпоративная почта. Сохраните её в словарь `MY_HEADERS` и отправьте с ним запрос к любой статье Википедии. Убедитесь, что статус-код равен 200 и сервер получил именно вашу подпись.

In [20]:
MY_HEADERS = {
    # ВАШ КОД
}

# ВАШ КОД

#### ✏️ Задание 3

Отправьте запрос со своими заголовками `MY_HEADERS` к странице, которой в Википедии нет (придумайте название). Выведите статус-код. Затем напишите условие: если код равен 200, печатается «Страница найдена», иначе — «Ошибка» и её код. Попробуйте отработать это на нескольких URL.

In [21]:
links = []

# ВАШ КОД

---
## Часть 4. Правила сайта

Технически скрипт может отправить тысячи запросов в минуту. Но каждый запрос — это нагрузка на чужой сервер, а собранные данные принадлежат не исследователю. Прежде чем собирать данные с сайта, стоит проверить три вещи.

**1. Файл robots.txt.** В корне большинства сайтов лежит файл `/robots.txt` с инструкциями для автоматических программ: какие разделы им можно посещать, а какие нет. Строка `User-agent` указывает, к каким программам относится правило (`*` — ко всем), строки `Disallow` и `Allow` — какие пути запрещены и разрешены. Юридической силы файл не имеет, но это прямо выраженная воля владельца сайта.

In [22]:
# robots.txt лежит в корне сайта: к адресу сайта добавляется /robots.txt
robots = requests.get("https://ru.wikipedia.org/robots.txt", headers=HEADERS, timeout=10)

robots.status_code

200

In [23]:
print(robots.text[:500])

# robots.txt for http://www.wikipedia.org/ and friends
#
# Please note: There are a lot of pages on this site, and there are
# some misbehaved spiders out there that go _way_ too fast. If you're
# irresponsible, your access to the site may be blocked.
#

# Observed spamming large amounts of https://en.wikipedia.org/?curid=NNNNNN
# and ignoring 429 ratelimit responses, claims to respect robots:
# http://mj12bot.com/
User-agent: MJ12bot
Disallow: /

# advertising-related bots:
User-agent: Mediapa


Строки, которые начинаются с `#`, — комментарии для людей. Файл Википедии длинный, поэтому выводится только его начало.

#### ✏️🔁📨 Задание 4

Обратитесь к разным известным вам сайтам в поисках файла `robots.txt`. Отправьте к ним запросы, получите ответ и сохраните файлы (вспомните, как это делать).

📨 Отправьте в телеграм-чат курса свой код и список сайтов: у каких `robots.txt` нашёлся, а у каких нет.

In [24]:
links = []

# ВАШ КОД

#### ✏️ Задание 5

Разбейте текст `robots.txt` Википедии на строки методом `.splitlines()` и посчитайте, сколько строк начинается с `Disallow`. Выведите первые пять таких строк.

In [25]:
# ВАШ КОД

**2. Пользовательское соглашение** (Terms of Use). В нём бывает прямо запрещён автоматический сбор, и тогда нарушение может иметь последствия.

**3. API.** Если у сайта есть API, данные лучше брать через него: это способ, который владелец сайта предусмотрел для программ. Что такое API и как узнать, есть ли он у сайта, — в следующей части.

И общие правила вежливого сбора:
- делать паузу между запросами; секунда — разумный минимум, если сайт не указал другого;
- отправлять запросы по одному, а не параллельно;
- не скачивать одно и то же дважды: сохранять полученное в файл;
- представляться в User-Agent.

---
## Часть 5. Что такое API

### Данные со страницы, сделанной для человека

Допустим, нужно узнать, сколько статей Википедии упоминают Пушкина. В браузере это просто: ввести слово в поиск, и над результатами появится строка «Результаты 1—20 из …». Тот же поиск можно отправить из Python: это обычная страница, у которой поисковое слово передаётся в параметре `search`.

In [26]:
# Тот же адрес, что открывается в браузере после поиска
search_page = requests.get(
    "https://ru.wikipedia.org/w/index.php",
    params={"search": "Пушкин", "fulltext": 1, "ns0": 1},
    headers=HEADERS,
    timeout=10,
)

search_page.status_code

200

In [27]:
# Сколько символов в ответе
len(search_page.text)

112553

In [28]:
# Начало ответа: служебная разметка, а не результаты поиска
print(search_page.text[:1000])

<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-enabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-disabled vector-feature-custom-font-size-clientpref--excluded vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-not-available skin-thumbsize-clientpref-standard skin-underline-clientpref-browser" lang="ru" dir="ltr">
<head>
<meta charset="UTF-8">
<title>Результаты поиска «Пушкин» — Википедия</title>
<script>(function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-enabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-cl

Число найденных статей где-то внутри этого текста. В браузере оно стоит после слова «Результаты». Метод `.find()` возвращает позицию, с которой в строке начинается фрагмент.

In [29]:
position = search_page.text.find("Результаты")

position

698

In [30]:
# 150 символов начиная с найденной позиции
print(search_page.text[position:position + 150])

Результаты поиска «Пушкин» — Википедия</title>
<script>(function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-l


Как мы понимаем, в этой служебной разметке нет того, что мы искали. Нужное место можно найти по служебной метке `results-info`, которой разработчики Википедии пометили строку с числом результатов. Как находить такие метки в коде страницы, разбирается на занятии 3.

In [68]:
position = search_page.text.find("results-info")

print(search_page.text[position:position + 190])

results-info" data-mw-num-results-offset="0" data-mw-num-results-total="32028" data-mw-num-results-approximate-total="false">Результаты <strong>1—20</strong> из <strong>32&#160;028</strong><


Число нашлось, но в неудобном виде: оно обёрнуто в теги `<strong>`, а пробел между разрядами записан кодом `&#160;`. Чтобы получить из этого фрагмента число 32028, придётся писать отдельный код. Сама страница при этом занимает больше ста тысяч символов, из которых нужны несколько. Кроме того, такой способ ненадёжен: если разработчики изменят оформление страницы, код перестанет работать без предупреждения.

### Вход для программ

У многих сайтов, кроме страниц для людей, есть отдельный вход для программ — **API** (Application programming Interface). Клиент отправляет запрос по определённому адресу с определёнными параметрами, а сервер присылает только данные, без оформления, обычно в формате JSON.

| | HTML-страница | API |
|---|---|---|
| Для кого | для человека | для программы |
| Что приходит | разметка, текст, оформление | только данные |
| Как найти нужное | разбирать разметку | обратиться к полю по названию |
| Правила | неявные, могут измениться без предупреждения | описаны в документации |

### Как узнать, есть ли у сайта API

API обычно описан в разделе для разработчиков. Ссылку на него ищут:

- в подвале сайта: «Разработчикам», «Developers», «API», «Открытые данные»;
- через поисковик: запрос вида «название сайта API»;
- в каталогах открытых API.

У Википедии в подвале любой страницы есть ссылка «Разработчики». Она ведёт на портал [developer.wikimedia.org](https://developer.wikimedia.org/), а оттуда — к документации **MediaWiki API**: так называется API Википедии и других сайтов на движке MediaWiki.

Большинство современных API устроены по принципу **REST**: у каждого вида данных свой адрес (**эндпоинт**, endpoint), а уточнения передаются параметрами. Поэтому работа с API — это почти всегда чтение документации: какой эндпоинт нужен и какие параметры он принимает.

### MediaWiki API

У основного API Википедии один эндпоинт для всех запросов:

`https://ru.wikipedia.org/w/api.php`

То, что нужно получить, указывается в параметрах. Для поиска по статьям:

| Параметр | Значение | Смысл |
|---|---|---|
| `action` | `query` | получить данные |
| `list` | `search` | список результатов поиска |
| `srsearch` | `Пушкин` | что ищем |
| `srlimit` | `10` | сколько результатов вернуть |
| `format` | `json` | в каком формате прислать ответ |

Полный список параметров — в [документации](https://www.mediawiki.org/wiki/API:Search) и в [справке самого API](https://ru.wikipedia.org/w/api.php?action=help&modules=query%2Bsearch).

Параметры можно вписать в адрес вручную, но удобнее передать словарём в аргумент `params`: тогда `requests` соберёт адрес и закодирует кириллицу. Ниже — тот же поиск «Пушкин», но через API.

In [ ]:
# Эндпоинт: один адрес для всех запросов к API Википедии
API_URL = "https://ru.wikipedia.org/w/api.php"

# Параметры запроса — обычный словарь
params = {
    "action": "query",
    "list": "search",
    "srsearch": "Пушкин",
    "srlimit": 10,
    "format": "json",
}

params

{'action': 'query',
 'list': 'search',
 'srsearch': 'Пушкин',
 'srlimit': 10,
 'format': 'json'}

In [33]:
# Параметры передаются в аргументе params, заголовки — в headers
api_response = requests.get(API_URL, params=params, headers=HEADERS, timeout=10)

api_response.status_code

200

In [34]:
# Адрес, который requests собрал из API_URL и params
api_response.url

'https://ru.wikipedia.org/w/api.php?action=query&list=search&srsearch=%D0%9F%D1%83%D1%88%D0%BA%D0%B8%D0%BD&srlimit=10&format=json'

In [35]:
# Для сравнения: размер ответа API и размер страницы поиска
print(len(api_response.text))
print(len(search_page.text))

10676
112553


Адрес из последней строки можно открыть в браузере и увидеть тот же ответ.

#### ✏️ Задание 6

Отправьте к API запрос с другим поисковым словом и с `srlimit`, равным 5. Выведите статус-код и первые 300 символов ответа (`.text`).

In [36]:
# ВАШ КОД

## Часть 6. JSON

### Формат

**JSON** (JavaScript Object Notation) — текстовый формат обмена данными. Он почти совпадает с записью словарей и списков в Python:

| JSON | Python | Пример |
|---|---|---|
| объект `{}` | `dict` | `{"title": "Пушкин"}` |
| массив `[]` | `list` | `[1, 2, 3]` |
| строка | `str` | `"Пушкин"` |
| число | `int`, `float` | `42`, `3.14` |
| `true`, `false` | `True`, `False` | - |
| `null` | `None` | - |

Метод `.json()` превращает текст ответа в словари и списки Python.

### 🔁 Словари и списки

Прежде чем разбирать настоящий ответ, стоит вспомнить, как устроены словари и списки. Значение из словаря достаётся по ключу в квадратных скобках, элемент списка — по номеру, начиная с нуля. Если внутри словаря лежит список, а внутри списка — словари, обращения записываются подряд.

In [37]:
writer = {
    "name": "Антон Чехов",
    "born": 1860,
    "plays": ["Чайка", "Дядя Ваня", "Три сестры", "Вишнёвый сад"],
    "places": [{"city": "Таганрог", "years": "1860–1879"}, {"city": "Москва", "years": "1879–1892"}],
}

print(writer["name"])                 # значение по ключу
print(writer.keys())                  # все ключи
print(writer["plays"][0])             # первый элемент списка
print(writer["plays"][-1])            # последний элемент списка
print(len(writer["plays"]))           # длина списка
print(writer["places"][1]["city"])    # словарь внутри списка внутри словаря

Антон Чехов
dict_keys(['name', 'born', 'plays', 'places'])
Чайка
Вишнёвый сад
4
Москва


In [38]:
# Цикл по списку и f-строка: значения подставляются в фигурные скобки
for play in writer["plays"]:
    print(f"{writer['name']} — «{play}»")

Антон Чехов — «Чайка»
Антон Чехов — «Дядя Ваня»
Антон Чехов — «Три сестры»
Антон Чехов — «Вишнёвый сад»


Ответ API устроен так же, только уровней вложенности и ключей больше.

In [39]:
data = api_response.json()

type(data)

dict

In [40]:
data.keys()

dict_keys(['batchcomplete', 'continue', 'query'])

Ответ — вложенная структура: словарь, внутри которого словари и списки. Чтобы добраться до нужного значения, ключи и индексы записываются цепочкой, по одному на каждый уровень вложенности.

In [41]:
data["query"].keys()

dict_keys(['searchinfo', 'search'])

In [42]:
# Что лежит под ключом searchinfo
data["query"]["searchinfo"]

{'totalhits': 32028}

In [43]:
# Сколько всего статей нашлось: три ключа подряд, по одному на уровень
data["query"]["searchinfo"]["totalhits"]

32028

In [44]:
# Результаты поиска — список словарей, по одному на статью
results = data["query"]["search"]

print(type(results))
print(len(results))

<class 'list'>
10


In [45]:
# Первая статья — первый элемент списка, то есть словарь
results[0]

{'ns': 0,
 'title': 'Пушкин, Александр Сергеевич',
 'pageid': 537,
 'size': 329510,
 'wordcount': 18954,
 'snippet': 'Алекса́ндр Серге́евич <span class="searchmatch">Пу́шкин</span> (26\xa0мая [6\xa0июня]\xa01799, Москва\xa0— 29\xa0января [10\xa0февраля]\xa01837, Санкт-Петербург)\xa0— русский поэт, драматург и прозаик, заложивший',
 'timestamp': '2026-10-10T12:18:13Z'}

In [46]:
# Одно поле первой статьи
results[0]["title"]

'Пушкин, Александр Сергеевич'

У каждого результата одинаковый набор полей:

| Поле | Что содержит |
|---|---|
| `title` | название статьи |
| `pageid` | числовой идентификатор статьи |
| `size` | размер статьи в байтах |
| `wordcount` | число слов |
| `snippet` | фрагмент текста вокруг найденного слова |
| `timestamp` | время последней правки |

Отдельно стоит ключ `continue` на верхнем уровне ответа: он означает, что результатов больше, чем пришло, а в его содержимом указано, как запросить следующую порцию. Об этом — на занятии 2.

#### ✏️ Задание 7

Пройдите циклом по списку `results` и выведите название каждой статьи и число слов в ней в виде `Пушкин, Александр Сергеевич — 12345 слов`.

In [47]:
# ВАШ КОД

#### ✏️ Задание 8

Найдите среди результатов самую длинную статью (по `wordcount`) и выведите её название.

*Подсказка:* можно перебрать список циклом, запоминая лучший результат, а можно воспользоваться функцией `max()` с аргументом `key`.

In [48]:
# ВАШ КОД

### 🔁 Регулярные выражения: `re.sub`

Функция `re.sub(шаблон, замена, строка)` заменяет все фрагменты строки, подходящие под шаблон. Точка в шаблоне означает любой символ, `*` — сколько угодно повторений предыдущего символа, а `?` после `*` делает повторение нежадным: захватывается как можно меньше символов.

In [49]:
text = "Вишнёвый [1903] сад [пьеса]"

print(re.sub(r"\d", "#", text))       # каждая цифра заменяется на #
print(re.sub(r"\[.*\]", "", text))    # жадно: от первой [ до последней ]
print(re.sub(r"\[.*?\]", "", text))   # нежадно: каждая пара скобок отдельно

Вишнёвый [####] сад [пьеса]
Вишнёвый 
Вишнёвый  сад 


#### ✏️ Задание 9

Поле `snippet` содержит HTML-разметку: найденное слово обёрнуто в тег `<span class="searchmatch">`. Напишите функцию `clean_snippet()`, которая удаляет из строки все теги с помощью регулярного выражения (как на первом занятии курса «Цифровые технологии в профессиональной деятельности»). Примените её к сниппету первого результата.

*Подсказка:* тег — это всё, что находится между `<` и ближайшим `>`.

In [50]:
def clean_snippet(snippet: str) -> str:
    # ВАШ КОД
    ...


print(results[0]["snippet"])
print(clean_snippet(results[0]["snippet"]))

Алекса́ндр Серге́евич <span class="searchmatch">Пу́шкин</span> (26 мая [6 июня] 1799, Москва — 29 января [10 февраля] 1837, Санкт-Петербург) — русский поэт, драматург и прозаик, заложивший
None


---
## Часть 7. Из JSON в таблицу

Список одинаково устроенных словарей превращается в таблицу одной строкой: каждый словарь становится строкой таблицы, а его ключи — столбцами.

### 🔁 pandas: что понадобится

| Запись | Что делает |
|---|---|
| `pd.DataFrame(список_словарей)` | таблица из списка словарей |
| `df.head()` | первые пять строк |
| `df[["a", "b"]]` | только столбцы `a` и `b` |
| `df["a"].apply(функция)` | применить функцию к каждому значению столбца |
| `df.sort_values("a", ascending=False)` | сортировка по столбцу, от большего к меньшему |
| `df[df["a"] > 100]` | только строки, где условие выполняется |

In [51]:
df = pd.DataFrame(results)

df.head()

,ns,title,pageid,size,wordcount,snippet,timestamp
0,0,"Пушкин, Александр Сергеевич",537,329510,18954,"Алекса́ндр Серге́евич <span class=""searchmatch...",2026-10-10T12:18:13Z
1,0,Пушкин (город),46524,213621,12274,"<span class=""searchmatch"">Пу́шкин</span> (до 1...",2026-09-02T19:50:26Z
2,0,Бобрищев-Пушкин,1307425,836,108,"Бобрищев-<span class=""searchmatch"">Пушкин</spa...",2026-03-31T18:57:50Z
3,0,Потомки Пушкина,695747,26023,1444,"музее А. С. <span class=""searchmatch"">Пушкина<...",2026-10-27T13:04:11Z
4,0,Пушкины,299981,30544,1891,"<span class=""searchmatch"">Пу́шкины</span> — др...",2026-10-16T18:14:50Z


In [52]:
# Какие столбцы получились: по одному на каждый ключ словаря
df.columns

Index(['ns', 'title', 'pageid', 'size', 'wordcount', 'snippet', 'timestamp'], dtype='object')

In [53]:
# Остаются только нужные столбцы; .copy() создаёт отдельную таблицу, а не ссылку на часть старой
df = df[["title", "pageid", "wordcount", "snippet", "timestamp"]].copy()

df.head()

,title,pageid,wordcount,snippet,timestamp
0,"Пушкин, Александр Сергеевич",537,18954,"Алекса́ндр Серге́евич <span class=""searchmatch...",2026-10-10T12:18:13Z
1,Пушкин (город),46524,12274,"<span class=""searchmatch"">Пу́шкин</span> (до 1...",2026-09-02T19:50:26Z
2,Бобрищев-Пушкин,1307425,108,"Бобрищев-<span class=""searchmatch"">Пушкин</spa...",2026-03-31T18:57:50Z
3,Потомки Пушкина,695747,1444,"музее А. С. <span class=""searchmatch"">Пушкина<...",2026-10-27T13:04:11Z
4,Пушкины,299981,1891,"<span class=""searchmatch"">Пу́шкины</span> — др...",2026-10-16T18:14:50Z


In [54]:
# Сниппеты очищаются: функция clean_snippet применяется к каждому значению столбца
df["snippet"] = df["snippet"].apply(clean_snippet)

df["snippet"].head()

0    None
1    None
2    None
3    None
4    None
Name: snippet, dtype: object

In [69]:
# Время правки пришло строкой (тип object)
print(df["timestamp"].dtype)

datetime64[ns, UTC]


In [ ]:
# Строка превращается в дату, чтобы потом извлекать из неё год, месяц и т. п.
df["timestamp"] = pd.to_datetime(df["timestamp"])
print(df["timestamp"].dtype)

In [56]:
df.head()

,title,pageid,wordcount,snippet,timestamp
0,"Пушкин, Александр Сергеевич",537,18954,None,2026-10-10 12:18:13+00:00
1,Пушкин (город),46524,12274,None,2026-09-02 19:50:26+00:00
2,Бобрищев-Пушкин,1307425,108,None,2026-03-31 18:57:50+00:00
3,Потомки Пушкина,695747,1444,None,2026-10-27 13:04:11+00:00
4,Пушкины,299981,1891,None,2026-10-16 18:14:50+00:00


Если функция `clean_snippet()` в задании 9 не дописана, в столбце `snippet` окажется `None`: вернитесь к заданию и перезапустите ячейку.

#### ✏️ Задание 10

Отсортируйте таблицу по числу слов от большего к меньшему и выведите три самые длинные статьи. Затем выведите статьи, которые правились в текущем году.

*Подсказка:* год из столбца с датами извлекается так: `df["timestamp"].dt.year`.

In [57]:
# ВАШ КОД

---
## Часть 8. Сохранение в CSV

Собранные данные нужно сразу сохранять: тогда для дальнейшей работы не придётся снова обращаться к серверу. Самый простой формат для таблиц — **CSV** (comma-separated values): обычный текстовый файл, где значения разделены запятыми.

In [58]:
df.to_csv("pushkin_search.csv", index=False, encoding="utf-8")

- `index=False` — не записывать в файл служебный индекс `pandas`;
- `encoding="utf-8"` — кодировка, в которой корректно сохраняется кириллица.

Excel иногда открывает такой файл с искажённой кириллицей. Если таблицу предстоит открывать в Excel, используйте `encoding="utf-8-sig"`.

Проверка: файл читается обратно.

In [59]:
pd.read_csv("pushkin_search.csv").head()

,title,pageid,wordcount,snippet,timestamp
0,"Пушкин, Александр Сергеевич",537,18954,NaN,2026-10-10 12:18:13+00:00
1,Пушкин (город),46524,12274,NaN,2026-09-02 19:50:26+00:00
2,Бобрищев-Пушкин,1307425,108,NaN,2026-03-31 18:57:50+00:00
3,Потомки Пушкина,695747,1444,NaN,2026-10-27 13:04:11+00:00
4,Пушкины,299981,1891,NaN,2026-10-16 18:14:50+00:00


---
## Часть 9. Несколько запросов подряд

Пока все запросы отправлялись вручную. Когда запросов много, код запроса удобно оформить функцией и вызывать в цикле, не забывая о паузе между запросами.

### 🔁 Функции

Функция объявляется словом `def`. В скобках перечисляются аргументы; у аргумента может быть значение по умолчанию, тогда при вызове его можно не указывать. Команда `return` возвращает результат. Строка в тройных кавычках сразу после объявления — описание функции.

In [60]:
def make_title(name: str, work: str, year: int | None = None) -> str:
    """Составляет подпись вида «Автор. Произведение (год)»."""
    if year is None:
        return f"{name}. {work}"
    return f"{name}. {work} ({year})"


print(make_title("А. П. Чехов", "Чайка", 1896))
print(make_title("А. П. Чехов", "Остров Сахалин"))  # год не указан — значение по умолчанию

А. П. Чехов. Чайка (1896)
А. П. Чехов. Остров Сахалин


Функция `search_wikipedia()` ниже устроена так же: аргументы — поисковый запрос и число результатов, результат — ответ API в виде словаря.

Метод `.raise_for_status()` прерывает выполнение с ошибкой, если статус-код начинается с 4 или 5. Без него программа продолжила бы работу с пустым или ошибочным ответом и сломалась бы позже, в менее понятном месте.

In [61]:
def search_wikipedia(query: str, limit: int = 10) -> dict:
    """Ищет статьи в русской Википедии и возвращает ответ API в виде словаря."""

    # Те же параметры, что в части 5, но слово и число результатов берутся из аргументов
    params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "srlimit": limit,
        "format": "json",
    }

    response = requests.get(API_URL, params=params, headers=HEADERS, timeout=10)
    response.raise_for_status()  # код 4xx или 5xx — остановиться с ошибкой

    time.sleep(1)  # пауза, чтобы не перегружать сервер
    
    return response.json()

In [62]:
# Проверка функции на одном запросе
answer = search_wikipedia("Пушкин", limit=1)

answer["query"]["searchinfo"]

{'totalhits': 32028}

In [63]:
writers = ["Пушкин", "Лермонтов", "Гоголь", "Тургенев", "Достоевский"]

rows = []  # сюда собираются строки будущей таблицы

for writer in writers:
    # limit=1: сами статьи не нужны, нужно только общее число
    answer = search_wikipedia(writer, limit=1)
    total = answer["query"]["searchinfo"]["totalhits"]

    # Каждая строка таблицы — словарь: ключи станут столбцами
    rows.append({"query": writer, "totalhits": total})
    print(writer, total)

# Таблица из списка словарей, отсортированная по убыванию числа статей
writers_df = pd.DataFrame(rows).sort_values("totalhits", ascending=False)
writers_df

Пушкин 32028
Лермонтов 1987
Гоголь 10343
Тургенев 1757
Достоевский 6954


,query,totalhits
0,Пушкин,32028
2,Гоголь,10343
4,Достоевский,6954
1,Лермонтов,1987
3,Тургенев,1757


Получилась маленькая культуромическая таблица: сколько статей Википедии упоминают каждого писателя. Прежде чем делать из неё выводы, стоит понять, что посчитано. В выдачу попадают статьи, где встречается слово, а не статьи о человеке: в счёт идут однофамильцы, города, улицы, пароходы, премии и музеи, названные в честь писателя. Кроме того, при поиске учитываются разные формы слова, поэтому по короткой и частой фамилии находится больше случайных совпадений.

#### ✏️📨 Задание 11

Посмотрите на заголовки первых 10 статей, найденных по запросу «Пушкин», — какие из них не о поэте? Затем уточните запрос: в поиске Википедии можно искать фразу в кавычках, например `'"Александр Пушкин"'`. Сравните `totalhits` для фамилии и для фразы. Какой из двух показателей лучше подходит для сравнения писателей между собой и чем плох каждый из них?

📨 Отправьте в телеграм-чат курса оба числа и ответ на последний вопрос в двух-трёх предложениях.

In [64]:
# ВАШ КОД

#### ✏️📨 Задание 12

Составьте собственный список из 5–7 имён или понятий из вашей области (философы, художники, исторические события, города) и соберите по нему такую же таблицу. Сохраните её в CSV. Запишите в ячейке Markdown ниже, что, по-вашему, на самом деле можно узнать из числа `totalhits` в вашем случае. 

📨 Отправьте в телеграм-чат курса таблицу и своё пояснение.

In [65]:
# ВАШ КОД

*Ваш ответ:*

---
## Часть 10. Самостоятельная работа: статьи из категории

Поиск — не единственная возможность API. Например, запрос `list=categorymembers` возвращает список статей из категории Википедии.

| Параметр | Значение | Смысл |
|---|---|---|
| `action` | `query` | получить данные |
| `list` | `categorymembers` | участники категории |
| `cmtitle` | `Категория:…` | название категории вместе с префиксом |
| `cmtype` | `page` | только статьи, без подкатегорий и файлов |
| `cmlimit` | `max` | максимум за один запрос (500) |
| `format` | `json` | формат ответа |

Документация: [API:Categorymembers](https://www.mediawiki.org/wiki/API:Categorymembers).

#### ✏️📨 Задание 13

1. Откройте в браузере любую статью Википедии из вашей области и найдите внизу страницы её категории. Выберите категорию и скопируйте её название вместе со словом «Категория:».
2. Напишите функцию `get_category_members(category)`, которая отправляет запрос `categorymembers` и возвращает список статей (`["query"]["categorymembers"]`). Не забудьте про User-Agent, `timeout`, `raise_for_status()` и паузу.
3. Превратите результат в таблицу, выведите число статей и сохраните таблицу в CSV.
4. Проверьте, есть ли в ответе ключ `continue`. Если есть, значит, статей в категории больше, чем пришло за один запрос. Как получить остальные — тема следующего занятия.

📨 Отправьте в телеграм-чат курса название категории, число статей и одну фразу о том, для какого исследования мог бы пригодиться такой список.

In [66]:
def get_category_members(category: str) -> list[dict]:
    # ВАШ КОД
    ...


# ВАШ КОД

---
## Итоги

Сегодня разобрано, как устроен обмен данными в вебе, и пройден полный путь от запроса до таблицы:

`URL и параметры` → `requests.get()` → `статус-код` → `.json()` → `словари и списки` → `DataFrame` → `CSV`

С его помощью собраны два исследовательских набора данных: таблица упоминаний для количественного сравнения и список статей категории как заготовка корпуса. Главный методологический вывод занятия: число из API — результат конкретного запроса к конкретной системе, и прежде чем его интерпретировать, нужно понять, что за ним стоит.

На следующем занятии путь усложнится: речь пойдёт о том, как читать документацию API, как получать данные порциями (пагинация), что делать с ошибками и лимитами, как работать с API, которые требуют ключ.

### Документация

- requests: https://requests.readthedocs.io/
- MediaWiki API: https://www.mediawiki.org/wiki/API:Main_page
- Правила User-Agent для проектов Wikimedia: https://meta.wikimedia.org/wiki/User-Agent_policy
- pandas, чтение и запись CSV: https://pandas.pydata.org/docs/user_guide/io.html#csv-text-files
- urllib.parse: https://docs.python.org/3/library/urllib.parse.html

### Полезное

- M. Walsh, *Introduction to Cultural Analytics & Python*, глава об API: https://melaniewalsh.github.io/Intro-Cultural-Analytics/
- Справка MediaWiki API с примерами запросов прямо в браузере: https://ru.wikipedia.org/wiki/Служебная:ApiSandbox